# 📄 Resume Classification Using NLP
**Internship Project | End-to-End Machine Learning Pipeline**

---

## 🎯 Objective
Build a machine learning model that classifies resumes into job roles using Natural Language Processing (NLP).

## 📦 Dataset
- **Source:** Kaggle Resume Dataset (`UpdatedResumeDataSet.csv`)
- **Columns:** `Category` (job role), `Resume` (raw text)
- **Size:** 962 resumes across 25 job categories

## 🗺️ Workflow
```
Load Data → EDA → Preprocess → TF-IDF → Split → Train → Evaluate → Tune → Save → Deploy
```

In [ ]:
# ──────────────────────────────────────────
# CELL 1 — Import Libraries
# ──────────────────────────────────────────
# We import all required libraries upfront.
# scikit-learn handles ML tasks; pandas/numpy handle data; matplotlib/seaborn handle charts.

import re
import os
import warnings
import joblib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

warnings.filterwarnings('ignore')
print('✅ All libraries imported successfully!')

---
## Step 1 — Load Dataset & EDA
EDA (Exploratory Data Analysis) helps us understand the structure, quality, and distribution of our data before modelling.

In [ ]:
# ──────────────────────────────────────────
# CELL 2 — Load the Dataset
# ──────────────────────────────────────────
# pd.read_csv() reads a CSV file into a pandas DataFrame.
# A DataFrame is like an Excel spreadsheet in Python.

df = pd.read_csv('../Resume.csv')   # adjust path if needed

print('Dataset Shape:', df.shape)   # (rows, columns)
print('Columns     :', df.columns.tolist())
print()
print('First 2 rows:')
df.head(2)

In [ ]:
# ──────────────────────────────────────────
# CELL 3 — Class Distribution
# ──────────────────────────────────────────
# Understanding how many resumes exist per job category.
# Imbalanced data (very unequal counts) can bias the model.

print('Category Distribution:')
print(df['Category'].value_counts())
print()
print('Missing Values:')
print(df.isnull().sum())

In [ ]:
# ──────────────────────────────────────────
# CELL 4 — Visualise Class Distribution
# ──────────────────────────────────────────
plt.figure(figsize=(14, 6))
order = df['Category'].value_counts().index
sns.countplot(data=df, y='Category', order=order, palette='viridis')
plt.title('Resume Category Distribution', fontsize=14, fontweight='bold')
plt.xlabel('Count')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

---
## Step 2 — Text Preprocessing

Raw resumes contain noise: URLs, emails, phone numbers, punctuation, stopwords. We remove all of these before feeding text to the model.

**Pipeline:**
1. Lowercase → 2. Remove URLs → 3. Remove Emails → 4. Remove Phones → 5. Remove Special Chars → 6. Remove Stopwords → 7. Filter short tokens

In [ ]:
# ──────────────────────────────────────────
# CELL 5 — Text Cleaning Function
# ──────────────────────────────────────────
# re.sub(pattern, replacement, text) replaces regex matches with replacement.
# ENGLISH_STOP_WORDS is a built-in set of 318 common English words to ignore.

def clean_text(text: str) -> str:
    """Full NLP text cleaning pipeline for resume text."""
    # 1. Lowercase — standardises all text
    text = text.lower()
    
    # 2. Remove URLs (http/https/www)
    text = re.sub(r'http\S+|www\S+|https\S+', ' ', text, flags=re.MULTILINE)
    
    # 3. Remove email addresses (anything@anything)
    text = re.sub(r'\S+@\S+', ' ', text)
    
    # 4. Remove phone numbers (7+ digit sequences with separators)
    text = re.sub(r'\b(\+?\d[\d\s\-().]{7,}\d)\b', ' ', text)
    
    # 5. Remove punctuation and special characters — keep only a-z and spaces
    text = re.sub(r'[^a-z\s]', ' ', text)
    
    # 6. Collapse multiple spaces into one
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 7. Tokenise and remove stopwords + short tokens
    tokens = [
        word for word in text.split()
        if word not in ENGLISH_STOP_WORDS and len(word) > 2
    ]
    
    return ' '.join(tokens)


# Test the function
sample = df['Resume'].iloc[0]
print('BEFORE CLEANING (100 chars):')
print(sample[:100])
print()
print('AFTER CLEANING (100 chars):')
print(clean_text(sample)[:100])

In [ ]:
# ──────────────────────────────────────────
# CELL 6 — Apply Cleaning to All Resumes
# ──────────────────────────────────────────
# apply() runs the function on every row in the column.
# This creates a new column 'cleaned_resume'.

df['cleaned_resume'] = df['Resume'].apply(clean_text)
print('✅ Cleaning applied to all', len(df), 'resumes')
print()
print('Average word count after cleaning:')
avg_words = df['cleaned_resume'].apply(lambda x: len(x.split())).mean()
print(f'  {avg_words:.0f} words per resume')

---
## Step 3 — Label Encoding

ML models cannot work with text labels like `"Data Science"`. We convert them to integers using `LabelEncoder`:
- `"Data Science"` → `3`
- `"Java Developer"` → `7`
- etc.

In [ ]:
# ──────────────────────────────────────────
# CELL 7 — Encode Category Labels
# ──────────────────────────────────────────

le = LabelEncoder()
df['label'] = le.fit_transform(df['Category'])

print(f'Total unique categories: {len(le.classes_)}')
print()
# Show the mapping for first 5 categories
mapping_df = pd.DataFrame({
    'Category': le.classes_,
    'Encoded Label': le.transform(le.classes_)
})
print(mapping_df.head(10).to_string(index=False))

---
## Step 4 — TF-IDF Feature Engineering

**What is TF-IDF?**

TF-IDF converts text into a matrix of numbers. It measures how *important* a word is in a document relative to the entire collection.

- **TF (Term Frequency):** How often a word appears in the document
  > `TF(t,d) = count(t in d) / total_words(d)`

- **IDF (Inverse Document Frequency):** How rare the word is across all documents
  > `IDF(t) = log( N / df(t) )` where N = total docs, df(t) = docs containing t

- **TF-IDF = TF × IDF** → High for rare but frequent words in a doc → good discriminator

In [ ]:
# ──────────────────────────────────────────
# CELL 8 — TF-IDF Vectorisation
# ──────────────────────────────────────────
# Parameters explained:
#   max_features=15000 : keep top 15k words (reduces noise from rare words)
#   ngram_range=(1,2)  : use single words AND 2-word phrases (e.g. 'machine learning')
#   min_df=2           : ignore words appearing in fewer than 2 documents
#   max_df=0.85        : ignore words in >85% of docs (too common to be useful)
#   sublinear_tf=True  : apply log(1+tf) to dampen high-frequency terms

tfidf = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=2,
    max_df=0.85,
    sublinear_tf=True
)

X = tfidf.fit_transform(df['cleaned_resume'])
y = df['label'].values

print(f'TF-IDF Matrix Shape : {X.shape}')
print(f'  → {X.shape[0]} resumes × {X.shape[1]} features')
print()
print('Top 10 features (vocabulary sample):')
print(list(tfidf.get_feature_names_out()[:10]))

---
## Step 5 — Train / Test Split

We split data 80/20: 80% for training, 20% for testing. `stratify=y` ensures each class appears proportionally in both splits.

In [ ]:
# ──────────────────────────────────────────
# CELL 9 — Stratified Train-Test Split
# ──────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,        # 20% for testing
    random_state=42,      # reproducibility seed
    stratify=y            # keep class proportions
)

print(f'Training samples : {X_train.shape[0]}')
print(f'Testing  samples : {X_test.shape[0]}')

---
## Step 6 — Model Training

We train two models:
1. **Logistic Regression** — linear classifier that uses sigmoid/softmax to output probabilities
2. **Multinomial Naive Bayes** — probabilistic classifier based on Bayes' theorem, excellent for text

In [ ]:
# ──────────────────────────────────────────
# CELL 10 — Train Models
# ──────────────────────────────────────────

# Logistic Regression
print('🔵 Training Logistic Regression...')
lr_model = LogisticRegression(
    max_iter=1000,
    C=1.0,                    # regularisation strength (smaller C = more regularisation)
    solver='lbfgs',           # optimisation algorithm
    # for multi-class classification
    random_state=42
)
lr_model.fit(X_train, y_train)
print('  ✅ Done!')

# Multinomial Naive Bayes
print('\n🟢 Training Multinomial Naive Bayes...')
nb_model = MultinomialNB(alpha=0.1)  # alpha = Laplace smoothing
nb_model.fit(X_train, y_train)
print('  ✅ Done!')

---
## Step 7 — Model Evaluation

**Key Metrics:**
- **Accuracy** = Correct predictions / Total predictions
- **Precision** = True Positives / (True Positives + False Positives) → How exact the model is
- **Recall** = True Positives / (True Positives + False Negatives) → How complete the model is
- **F1-Score** = 2 × (Precision × Recall) / (Precision + Recall) → Harmonic mean of P & R
- **Confusion Matrix** = Grid showing actual vs predicted for each class

In [ ]:
# ──────────────────────────────────────────
# CELL 11 — Evaluation Helper Function
# ──────────────────────────────────────────

def evaluate_model(model, model_name, X_train, X_test, y_train, y_test):
    y_pred = model.predict(X_test)
    acc    = accuracy_score(y_test, y_pred)
    prec   = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec    = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1     = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    
    print(f'\n{'='*55}')
    print(f'  {model_name}')
    print(f'{'='*55}')
    print(f'  Accuracy  : {acc:.4f}')
    print(f'  Precision : {prec:.4f}')
    print(f'  Recall    : {rec:.4f}')
    print(f'  F1-Score  : {f1:.4f}')
    
    print(f'\n  Classification Report:')
    print(classification_report(y_test, y_pred, target_names=le.classes_, zero_division=0))
    
    # Cross-validation
    cv = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    print(f'  CV Scores (5-fold): {cv.round(4)}')
    print(f'  Mean CV Accuracy  : {cv.mean():.4f} ± {cv.std():.4f}')
    
    # Confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    plt.figure(figsize=(18, 14))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Confusion Matrix — {model_name}', fontsize=13, fontweight='bold')
    plt.xlabel('Predicted'); plt.ylabel('Actual')
    plt.xticks(rotation=45, ha='right', fontsize=8)
    plt.yticks(fontsize=8)
    plt.tight_layout()
    plt.show()
    
    return {'Model': model_name, 'Accuracy': acc, 'Precision': prec,
            'Recall': rec, 'F1': f1, 'CV Mean': cv.mean()}


results = []
results.append(evaluate_model(lr_model, 'Logistic Regression', X_train, X_test, y_train, y_test))
results.append(evaluate_model(nb_model,  'Multinomial Naive Bayes', X_train, X_test, y_train, y_test))

In [ ]:
# ──────────────────────────────────────────
# CELL 12 — Performance Comparison Table
# ──────────────────────────────────────────

comparison = pd.DataFrame(results).set_index('Model')
print('\nPERFORMANCE COMPARISON:')
print(comparison.round(4).to_string())

# Bar chart comparison
comparison[['Accuracy','Precision','Recall','F1']].plot(
    kind='bar', figsize=(10, 5), rot=0, colormap='viridis'
)
plt.title('Model Performance Comparison', fontsize=13, fontweight='bold')
plt.ylabel('Score')
plt.ylim(0.7, 1.0)
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

---
## Step 8 — Hyperparameter Tuning

**GridSearchCV** tries every combination of parameters we specify and picks the best one using cross-validation. This prevents overfitting to the training set.

In [ ]:
# ──────────────────────────────────────────
# CELL 13 — GridSearchCV Tuning
# ──────────────────────────────────────────

print('🔵 Tuning Logistic Regression...')
lr_grid = GridSearchCV(
    LogisticRegression(solver='lbfgs',  random_state=42),
    {'C': [0.01, 0.1, 1, 10], 'max_iter': [500, 1000]},
    cv=3, scoring='accuracy', n_jobs=-1
)
lr_grid.fit(X_train, y_train)
print(f'  Best Params: {lr_grid.best_params_}')
print(f'  Best CV Acc: {lr_grid.best_score_:.4f}')

print()
print('🟢 Tuning Naive Bayes...')
nb_grid = GridSearchCV(
    MultinomialNB(),
    {'alpha': [0.01, 0.05, 0.1, 0.5, 1.0]},
    cv=3, scoring='accuracy', n_jobs=-1
)
nb_grid.fit(X_train, y_train)
print(f'  Best Params: {nb_grid.best_params_}')
print(f'  Best CV Acc: {nb_grid.best_score_:.4f}')

lr_tuned = lr_grid.best_estimator_
nb_tuned = nb_grid.best_estimator_

In [ ]:
# ──────────────────────────────────────────
# CELL 14 — Evaluate Tuned Models
# ──────────────────────────────────────────

results.append(evaluate_model(lr_tuned, 'LR Tuned', X_train, X_test, y_train, y_test))
results.append(evaluate_model(nb_tuned, 'NB Tuned', X_train, X_test, y_train, y_test))

# Final comparison
df_all = pd.DataFrame(results).set_index('Model')
print('\nFINAL COMPARISON (Before vs After Tuning):')
print(df_all.round(4).to_string())

---
## Step 9 — Save Models

We use **Joblib** to serialize (pickle) the trained models to `.pkl` files. This allows us to reload them later without retraining.

In [ ]:
# ──────────────────────────────────────────
# CELL 15 — Save All Artifacts
# ──────────────────────────────────────────

os.makedirs('../saved_models', exist_ok=True)

# Save TF-IDF vectorizer
joblib.dump(tfidf, '../saved_models/tfidf.pkl')

# Save label encoder
joblib.dump(le, '../saved_models/label_encoder.pkl')

# Save best model (pick highest accuracy after tuning)
lr_acc = accuracy_score(y_test, lr_tuned.predict(X_test))
nb_acc = accuracy_score(y_test, nb_tuned.predict(X_test))
best   = lr_tuned if lr_acc >= nb_acc else nb_tuned
joblib.dump(best, '../saved_models/model.pkl')

print('💾 Saved:')
print('   saved_models/model.pkl')
print('   saved_models/tfidf.pkl')
print('   saved_models/label_encoder.pkl')

In [ ]:
# ──────────────────────────────────────────
# CELL 16 — Reload and Test Saved Model
# ──────────────────────────────────────────

model_loaded = joblib.load('../saved_models/model.pkl')
tfidf_loaded = joblib.load('../saved_models/tfidf.pkl')
le_loaded    = joblib.load('../saved_models/label_encoder.pkl')

# Test with a sample
test_resume = (
    'Python developer with Django Flask REST APIs PostgreSQL Docker '
    'Git microservices unit testing AWS Lambda SQLAlchemy pytest'
)
cleaned     = clean_text(test_resume)
vec         = tfidf_loaded.transform([cleaned])
label       = model_loaded.predict(vec)[0]
category    = le_loaded.inverse_transform([label])[0]
confidence  = model_loaded.predict_proba(vec)[0].max() * 100

print(f'Resume snippet : {test_resume[:60]}...')
print(f'Predicted role : {category}')
print(f'Confidence     : {confidence:.1f}%')

---
## Step 10 — Real Resume Testing

We test the final model on 5 sample resumes from different domains.

In [ ]:
# ──────────────────────────────────────────
# CELL 17 — Test 5 Sample Resumes
# ──────────────────────────────────────────

sample_resumes = {
    'Data Scientist': (
        'Experienced data scientist Python machine learning deep learning '
        'TensorFlow scikit-learn NLP data analysis pandas numpy statistical '
        'modeling A/B testing Tableau SQL Big Data Spark'
    ),
    'Python Developer': (
        'Python developer Django Flask REST APIs PostgreSQL Docker Git '
        'CI/CD pipelines microservices unit testing AWS Lambda Celery Redis '
        'SQLAlchemy FastAPI pytest'
    ),
    'HR': (
        'Human Resources professional talent acquisition employee relations '
        'payroll processing performance management onboarding HRIS systems '
        'labor law compliance compensation benchmarking training'
    ),
    'Java Developer': (
        'Java developer Spring Boot Hibernate Maven RESTful web services '
        'microservices MySQL Oracle JUnit Apache Kafka Docker Kubernetes '
        'CI/CD Agile SCRUM'
    ),
    'Testing': (
        'QA testing engineer Selenium WebDriver TestNG JUnit manual testing '
        'automation testing JIRA bug tracking performance testing JMeter '
        'API testing Postman regression testing'
    ),
}

print(f'{"─"*65}')
print(f'  {"Actual":<20} {"Predicted":<25} {"Confidence":>10}')
print(f'{"─"*65}')

for actual, text in sample_resumes.items():
    cleaned    = clean_text(text)
    vec        = tfidf_loaded.transform([cleaned])
    pred_label = model_loaded.predict(vec)[0]
    pred_cat   = le_loaded.inverse_transform([pred_label])[0]
    conf       = model_loaded.predict_proba(vec)[0].max() * 100
    match      = '✅' if actual == pred_cat else '❌'
    print(f'  {match} {actual:<20} {pred_cat:<25} {conf:>8.1f}%')

print(f'{"─"*65}')

---
## ✅ Project Summary

| Step | Task | Status |
|------|------|--------|
| 1 | Dataset loading & EDA | ✅ |
| 2 | Text preprocessing | ✅ |
| 3 | Label encoding | ✅ |
| 4 | TF-IDF feature engineering | ✅ |
| 5 | Train/Test split | ✅ |
| 6 | Model training (LR + NB) | ✅ |
| 7 | Evaluation (Acc, P, R, F1, CV) | ✅ |
| 8 | Hyperparameter tuning (GridSearchCV) | ✅ |
| 9 | Save artifacts (joblib) | ✅ |
| 10 | Real resume testing | ✅ |

**Next Step:** Run `streamlit run app.py` to launch the web UI.